In [10]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [11]:


data_dir = "/kaggle/input/covid19-pneumonia-normal-chest-xray-pa-dataset"
classes = ["covid", "normal", "pneumonia"]
img_size = (224, 224)
batch_size = 32



def preprocess_image(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    image = cv2.GaussianBlur(image, (5, 5), 0)    
    image = image / 255.0
    image = np.stack((image,) * 3, axis=-1)
    return image



def create_datagen():
    return ImageDataGenerator(
        preprocessing_function=preprocess_image,
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        validation_split=0.2
    )



In [12]:
datagen = create_datagen()

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)

Found 5523 images belonging to 3 classes.
Found 1379 images belonging to 3 classes.


In [13]:
plt.figure(figsize=(12,6))

for i in range(6):
    plt.subplot(2,3,i+1)
    plt.imshow(X[i])
    plt.title(classes[y[i]])
    plt.axis("off")

plt.show()

NameError: name 'plt' is not defined

In [10]:






















from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.models import Sequential

# Define the input layer
input_layer = Input(shape=(224, 224, 3))

# CNN Model Architecture
model = Sequential([
    input_layer,
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(classes), activation='softmax')
])

# Compile the Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Model Summary
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)                    │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_8 (MaxPooling2D)       │ (None, 111, 111, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_9 (Conv2D)                    │ (None, 109, 109, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_9 (MaxPooling2D)       │ (None, 54, 54, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 52, 52, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_10 (MaxPooling2D)      │ (None, 26, 26, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 24, 24, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_11 (MaxPooling2D)      │ (None, 12, 12, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 36864)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 128)                 │       4,718,720 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,107,523 (19.48 MB)

 Trainable params: 5,107,523 (19.48 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Training
epochs = 50
history = model.fit(train_generator, validation_data=val_generator, epochs=epochs)

Epoch 1/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 207s 1s/step - accuracy: 0.5941 - loss: 0.8554 - val_accuracy: 0.6193 - val_loss: 1.3811
Epoch 2/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 156s 905ms/step - accuracy: 0.7718 - loss: 0.5462 - val_accuracy: 0.6708 - val_loss: 1.2143
Epoch 3/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 161s 930ms/step - accuracy: 0.8266 - loss: 0.4381 - val_accuracy: 0.6824 - val_loss: 1.6720
Epoch 4/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 166s 961ms/step - accuracy: 0.8451 - loss: 0.3954 - val_accuracy: 0.7092 - val_loss: 1.5754
Epoch 5/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 150s 870ms/step - accuracy: 0.8613 - loss: 0.3515 - val_accuracy: 0.7121 - val_loss: 1.6673
Epoch 6/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 150s 868ms/step - accuracy: 0.8773 - loss: 0.3273 - val_accuracy: 0.7397 - val_loss: 1.8284
Epoch 7/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 149s 864ms/step - accuracy: 0.8883 - loss: 0.2920 - val_accuracy: 0.7404 - val_loss: 1.6564
Epoch 8/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 148s 857ms/step - accuracy: 0.8993 - lo

In [13]:
loss, accuracy = model.evaluate(val_generator)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

44/44 ━━━━━━━━━━━━━━━━━━━━ 32s 723ms/step - accuracy: 0.8557 - loss: 3.2848
Validation Accuracy: 84.26%


In [1]:
# Save the Model
model.save("chest_xray_model.h5")


NameError: name 'model' is not defined